In [ ]:
!nvidia-smi

Sat Apr 27 04:57:18 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7

In [ ]:

import os
import random
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from transformers import pipeline

In [ ]:

system_message  = """this is a medical raw text comes from OCR. This contains Medicine Type, Medicine Name, Medicine power such as *mg or **ml, * means a number, mention doses like 1+1+1 or something else, some times each medicine have mention how much time to intake such as (*) or simple *
Here ** means a number. Make sure that, a medicine name or medicine type can not be start with a number or symbol.

now organize this data according to this format
Medicine Type:
Medicine Name:
Medicine Dose:
Intake Time:

and also mention if there is any special case."""

print(system_message)


this is a medical raw text comes from OCR. This contains Medicine Type, Medicine Name, Medicine power such as *mg or **ml, * means a number, mention doses like 1+1+1 or something else, some times each medicine have mention how much time to intake such as (*) or simple *
Here ** means a number. Make sure that, a medicine name or medicine type can not be start with a number or symbol.

now organize this data according to this format
Medicine Type:
Medicine Name:
Medicine Dose:
Intake Time:

and also mention if there is any special case.


In [ ]:
import pandas as pd
csv_file_path = '/content/ds.csv'
df = pd.read_csv(csv_file_path)
print('Here are the first few rows of the DataFrame:')
df.head(10)


Here are the first few rows of the DataFrame:


,raw,formatted
0,Tab IDA 625mg (21)\r\n1+1+1\r\nTab lora 10mg (...,"{""medicines"": [{""type"": ""Tab"", ""name"": ""IDA 62..."
1,Tab IDA 625mg (21)\r\n1+1+1\r\nTab lora 10mg (...,"{""medicines"": [{""type"": ""Tab"", ""name"": ""IDA 62..."
2,mfelulena saline (1oooml)\r\nTaba saline (1ooo...,"{""medicines"": [\r\n {""type"": ""Saline"", ""name""..."
3,Tab IDA 625mg (21)\r\n1+1+1\r\nNapa 500mg(10)\...,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
4,Tab Napa 500mg (20mg)\r\n2+2+2\r\nap ometid 20...,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
5,Tab. UDyl 10mg\r\nTab Napa 500mg (10)\r\n1+1+1...,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
6,Tab - Fexo 120\r\n0+0+1\r\nsefyz 600\r\n1+0+1\...,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
7,Tab: Paracitamol\r\n1+1+1 - 2 days\r\nTab: pot...,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
8,Tab. Anlentin 20mg\r\nAncloth (6mg)\r\n2+0+1+1...,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
9,Tab. Capotid 50mg\r\nTab Meesto 20\r\nTab Peiscin,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."


In [ ]:
train_df = df.sample(frac=0.9, random_state=42)
test_df = df.drop(train_df.index)

In [ ]:
train_df.to_json('train.jsonl', orient='records', lines=True)
test_df.to_json('test.jsonl', orient='records', lines=True)

In [ ]:
train_df

,raw,formatted
9,Tab. Capotid 50mg\r\nTab Meesto 20\r\nTab Peiscin,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
11,"Tab NeoFlex 500\r\nTab Metro 400\r\ncap, omep 20","{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
0,Tab IDA 625mg (21)\r\n1+1+1\r\nTab lora 10mg (...,"{""medicines"": [{""type"": ""Tab"", ""name"": ""IDA 62..."
13,mfelulena saline (1oooml)\r\nTabna saline (1oo...,"{""medicines"": [\r\n {""type"": ""Saline"", ""name""..."
5,Tab. UDyl 10mg\r\nTab Napa 500mg (10)\r\n1+1+1...,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
8,Tab. Anlentin 20mg\r\nAncloth (6mg)\r\n2+0+1+1...,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."
2,mfelulena saline (1oooml)\r\nTaba saline (1ooo...,"{""medicines"": [\r\n {""type"": ""Saline"", ""name""..."
1,Tab IDA 625mg (21)\r\n1+1+1\r\nTab lora 10mg (...,"{""medicines"": [{""type"": ""Tab"", ""name"": ""IDA 62..."
14,cap op - (10)\r\ncap. Flux 20\r\n?T+1D - 0 + 1...,"{""medicines"": [\r\n {""type"": ""Cap"", ""name"": ""..."
4,Tab Napa 500mg (20mg)\r\n2+2+2\r\nap ometid 20...,"{""medicines"": [\r\n {""type"": ""Tab"", ""name"": ""..."


In [ ]:
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
dataset_name = "/content/train.jsonl"

new_model = "Medi-llama-2-7b-custom1000"

num_train_epochs = 20

device_map = {"": 0} ## LOAD THE ENTIRE MODLE ON THE GPU

In [ ]:
device_map

{'': 0}

In [ ]:
# Load datasets
train_dataset = load_dataset('json', data_files='/content/train.jsonl', split="train")
valid_dataset = load_dataset('json', data_files='/content/test.jsonl', split="train")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# Preprocess datasets
train_dataset_mapped = train_dataset.map(lambda examples: {'text': [f'[INST] <>\n{system_message.strip()}\n<>\n\n' + prompt + ' [/INST] ' + response for prompt, response in zip(examples['raw'], examples['formatted'])]}, batched=True)
valid_dataset_mapped = valid_dataset.map(lambda examples: {'text': [f'[INST] <>\n{system_message.strip()}\n<>\n\n' + prompt + ' [/INST] ' + response for prompt, response in zip(examples['raw'], examples['formatted'])]}, batched=True)


Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
train_dataset_mapped

Dataset({
    features: ['raw', 'formatted', 'text'],
    num_rows: 14
})

In [ ]:
# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
print(compute_dtype)

torch.float16


In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: write)

In [ ]:
def create_model_and_tokenizer():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        use_safetensors=True,
        quantization_config=bnb_config,
        trust_remote_code=True,
        device_map="auto",
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer

# call the function
model, tokenizer = create_model_and_tokenizer()
model.config.use_cache = False

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():

        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=64,
    # target_modules=["query_key_value"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], #specific to Llama models.
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print_trainable_parameters(model)

trainable params: 16777216 || all params: 3517190144 || trainable%: 0.477006226934315


In [ ]:
from transformers import TrainingArguments

output_dir = "./results"

training_arguments = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    logging_steps=2,
    learning_rate=1e-4,
    fp16=True,
    max_grad_norm=0.3,
    num_train_epochs=20,
    evaluation_strategy="steps",
    eval_steps=2,
    warmup_ratio=0.05,
    save_strategy="epoch",
    group_by_length=True,
    output_dir=output_dir,
    report_to="tensorboard",
    save_safetensors=True,
    lr_scheduler_type="cosine",
    seed=42,
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!

In [ ]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_mapped,
    eval_dataset=valid_dataset_mapped,  # Pass validation dataset here
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=None,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)

# Train model
trainer.train()

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:159: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
2,2.614600,2.444737
4,2.318600,2.100759
6,2.025800,1.824389
8,1.785400,1.562065
10,1.564400,1.329783
12,1.370000,1.164348
14,1.231000,1.051081
16,1.137500,0.985527
18,1.084900,0.955169
20,1.063200,0.948454


/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.

TrainOutput(global_step=20, training_loss=1.6195313453674316, metrics={'train_runtime': 483.9633, 'train_samples_per_second': 0.579, 'train_steps_per_second': 0.041, 'total_flos': 2850840587796480.0, 'train_loss': 1.6195313453674316, 'epoch': 20.0})

In [ ]:
peft_model_path="./polok-llama-2-7b-custom1000"

trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

('./polok-llama-2-7b-custom1000/tokenizer_config.json',
 './polok-llama-2-7b-custom1000/special_tokens_map.json',
 './polok-llama-2-7b-custom1000/tokenizer.model',
 './polok-llama-2-7b-custom1000/added_tokens.json',
 './polok-llama-2-7b-custom1000/tokenizer.json')

In [ ]:
from transformers import TextStreamer
model.config.use_cache = True
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear4bit(
                in_features=4096, out_features=4096, bias=False
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): Linear4bit(
                in_features=4096, out_features=4096, bias=False

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

peft_model_dir = "polok-llama-2-7b-custom1000"

# load trained LLM model and tokenizer
trained_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_dir,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    load_in_4bit=True,
)
tokenizer = AutoTokenizer.from_pretrained(peft_model_dir)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
input_text = """03+12023
Rx
Rx
Rx
Nebulization stat as 01
Vertoline Inhalles
02 pre opals madichine
Tab. Kadifen (10mg)
0+0+4
Tab Napa 500mg
1+1+1
1+0+1"""


prompt = f"[INST] <>\n{system_message}\n<>\n\n {input_text} [/INST]"
num_new_tokens = 1000

num_prompt_tokens = len(tokenizer(prompt)['input_ids'])
max_length = num_prompt_tokens + num_new_tokens

gen = pipeline('text-generation', model=trained_model, tokenizer=tokenizer, max_length=max_length)
result = gen(prompt)
print(result[0]['generated_text'].replace(prompt, ''))

The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'LlamaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MusicgenForCausalLM', 'MvpForCausalLM', 'OpenLlamaForCausalLM', 'OpenAIGPTLMHeadModel', 'OPTForCausalLM', 'PegasusForCausalLM', 'PLBartForCausalLM', 'ProphetNetForCausalLM', 'QDQBertLMHeadModel', 'ReformerModelWithLMHead', 'RemBertForCausal

 {3}

Medicine Type:

* Nebulization (01)

Medicine Name:

* Vertoline Inhalles (02)
* Tab. Kadifen (10mg) (0+0+4)
* Tab Napa 500mg (1+1+1)

Medicine Dose:

* Nebulization (01) - *
* Vertoline Inhalles (02) - *
* Tab. Kadifen (10mg) (0+0+4) - *
* Tab Napa 500mg (1+1+1) - *

Intake Time:

* Nebulization (01) - *
* Vertoline Inhalles (02) - *
* Tab. Kadifen (10mg) (0+0+4) - *
* Tab Napa 500mg (1+1+1) - *

Special Cases:

* There are multiple doses for the same medicine name.
* Some medicine names contain numbers or symbols.

Note:

* * means a number or symbol.
* N/A means not applicable.

Also, there are some medicine names which are not in proper format, I have mentioned them in the special cases section.


## Marged the base and trained model then load the model and inference the marged model

In [ ]:
# # Merge and save the fine-tuned model

# new_model = "/content/polok-llama-2-7b-custom1000"
# model_path = "llama2_finetune_marged"  # change to your preferred path

# # Reload model in FP16 and merge it with LoRA weights
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     low_cpu_mem_usage=True,
#     return_dict=True,
#     torch_dtype=torch.float16,
#     device_map=device_map,
# )
# model_m = PeftModel.from_pretrained(base_model, new_model)
# model_m = model_m.merge_and_unload()

# # Reload tokenizer to save it
# tokenizer_m = AutoTokenizer.from_pretrained(new_model, trust_remote_code=True)
# tokenizer_m.pad_token = tokenizer_m.eos_token
# tokenizer_m.padding_side = "right"

# # Save the merged model
# model_m.save_pretrained(model_path)
# tokenizer_m.save_pretrained(model_path)

In [ ]:
# from google.colab import drive
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model = AutoModelForCausalLM.from_pretrained(model_path)
# tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
# input_text = """03+12023
# Rx
# Rx
# Rx
# Nebulization stat as 01
# Vertoline Inhalles
# 02 pre opals madichine
# Tab. Kadifen (10mg)
# 0+0+4
# Tab Napa 500mg
# 1+1+1
# 1+0+1"""


# prompt = f"[INST] <>\n{system_message}\n<>\n\n {input_text} [/INST]"
# num_new_tokens = 1000

# num_prompt_tokens = len(tokenizer(prompt)['input_ids'])
# max_length = num_prompt_tokens + num_new_tokens

# gen = pipeline('text-generation', model=trained_model, tokenizer=tokenizer, max_length=max_length)
# result = gen(prompt)
# print(result[0]['generated_text'].replace(prompt, ''))